In [ ]:
!pip -q install pandas numpy matplotlib seaborn scipy statsmodels transformers sentencepiece

from google.colab import files
import io
import pandas as pd

uploaded = files.upload()
input_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[input_name]))
print('Строк в файле:', len(df))


## 1. Получаем тональность

Если меток еще нет, модель RuBERT определит для каждого поста одну из трех категорий: положительный, нейтральный или негативный. На GPU это выполняется заметно быстрее.


In [ ]:
import re

if 'sent_label' not in df.columns:
    import torch
    from transformers import pipeline

    stopwords = set('и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее мне было вот от меня еще нет о из ему теперь когда даже ну'.split())

    def clean_text(text):
        text = str(text).lower()
        text = re.sub(r'http\S+|www\.\S+', ' ', text)
        text = re.sub(r'[#@]\S+', ' ', text)
        text = re.sub(r'[^а-яa-zё\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return ' '.join(word for word in text.split() if word not in stopwords and len(word) > 2)

    df = df.dropna(subset=['Text']).copy()
    df['clean_text'] = df['Text'].apply(clean_text)
    df = df[df['clean_text'].str.len() > 2].copy()
    device = 0 if torch.cuda.is_available() else -1
    classifier = pipeline('sentiment-analysis', model='blanchefort/rubert-base-cased-sentiment', device=device)
    predictions = []
    texts = df['clean_text'].tolist()
    for start in range(0, len(texts), 32):
        predictions.extend(classifier(texts[start:start + 32], truncation=True, max_length=512))
        print(f'Обработано: {min(start + 32, len(texts))} из {len(texts)}')
    df['sent_label'] = [item['label'] for item in predictions]
    df['sent_score'] = [item['score'] for item in predictions]
else:
    print('Файл уже содержит sent_label, классификация пропущена.')

df.to_csv('vk_with_sentiment.csv', index=False)
print('Готово. Постов с меткой:', len(df))


## 2. Подготавливаем признаки времени и вовлечённости


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, kruskal, mannwhitneyu, spearmanr
from statsmodels.stats.proportion import proportions_ztest

day_names = {'Monday': 'Понедельник', 'Tuesday': 'Вторник', 'Wednesday': 'Среда', 'Thursday': 'Четверг', 'Friday': 'Пятница', 'Saturday': 'Суббота', 'Sunday': 'Воскресенье'}
day_order = list(day_names.values())
period_order = ['Ночь', 'Утро', 'День', 'Вечер']
sentiment_order = ['NEGATIVE', 'NEUTRAL', 'POSITIVE']

def get_period(hour):
    if hour in [23, 0, 1, 2, 3, 4]: return 'Ночь'
    if hour <= 11: return 'Утро'
    if hour <= 17: return 'День'
    return 'Вечер'

df['date_time'] = pd.to_datetime(df['Date & Time'], errors='coerce')
df = df.dropna(subset=['date_time', 'sent_label']).copy()
df = df[df['sent_label'].isin(sentiment_order)].drop_duplicates(subset=['Post Link']).copy()
df['day'] = df['date_time'].dt.day_name().map(day_names)
df['hour'] = df['date_time'].dt.hour
df['period'] = df['hour'].apply(get_period)
df['negative'] = (df['sent_label'] == 'NEGATIVE').astype(int)
df['sentiment_score'] = df['sent_label'].map({'NEGATIVE': -1, 'NEUTRAL': 0, 'POSITIVE': 1})
df['engagement'] = df[['Likes', 'Comments', 'Reposts']].fillna(0).sum(axis=1)
text_col = 'clean_text' if 'clean_text' in df.columns else 'Text'
df['word_count'] = df[text_col].fillna('').astype(str).str.split().str.len()
df['monday_evening'] = ((df['day'] == 'Понедельник') & df['hour'].between(18, 22)).astype(int)
print(f'После очистки: {len(df)} постов, {df["Author ID"].nunique()} авторов')


## 3. Общая картина


In [ ]:
sentiment_summary = df['sent_label'].value_counts().reindex(sentiment_order, fill_value=0).to_frame('posts')
sentiment_summary['share'] = sentiment_summary['posts'] / len(df)
display(sentiment_summary.style.format({'share': '{:.1%}'}))

sns.set_theme(style='whitegrid')
sentiment_summary['posts'].plot(kind='bar', color=['#c44536', '#7b8794', '#2f855a'], figsize=(7, 4))
plt.title('Распределение тональности')
plt.ylabel('Количество постов')
plt.xticks(rotation=0)
plt.savefig('sentiment_distribution.png', dpi=180, bbox_inches='tight')
plt.show()


## 4. Гипотеза 1: понедельник вечером — самый негативный период?


In [ ]:
target = df[df['monday_evening'] == 1]
other = df[df['monday_evening'] == 0]
target_negative = int(target['negative'].sum())
other_negative = int(other['negative'].sum())
target_share = target_negative / len(target)
other_share = other_negative / len(other)
_, p_value = proportions_ztest([target_negative, other_negative], [len(target), len(other)])
print(f'Понедельник вечером: {target_negative} из {len(target)} ({target_share:.1%})')
print(f'Остальное время: {other_negative} из {len(other)} ({other_share:.1%})')
print(f'Разница: {(target_share - other_share) * 100:.1f} процентного пункта')
print('Вывод: гипотеза не подтверждается.')


`p-value` — это проверка, могла ли разница возникнуть случайно. Малое значение усиливает аргумент в пользу различия. Здесь результат не поддерживает исходную гипотезу.


In [ ]:
weekday_summary = df.groupby('day', observed=False).agg(posts=('negative', 'size'), negative_share=('negative', 'mean')).reindex(day_order)
display(weekday_summary.style.format({'negative_share': '{:.1%}'}))
weekday_summary['negative_share'].mul(100).plot(kind='bar', color='#c44536', figsize=(9, 4))
plt.title('Доля негативных постов по дням недели')
plt.ylabel('Негативные посты, %')
plt.xticks(rotation=25)
plt.savefig('negative_by_weekday.png', dpi=180, bbox_inches='tight')
plt.show()

heatmap = df.pivot_table(index='day', columns='period', values='negative', aggfunc='mean').reindex(index=day_order, columns=period_order) * 100
counts = df.pivot_table(index='day', columns='period', values='negative', aggfunc='size').reindex(index=day_order, columns=period_order)
plt.figure(figsize=(9, 6))
sns.heatmap(heatmap, annot=counts, fmt='.0f', cmap='Reds', cbar_kws={'label': '% негативных'})
plt.title('День и период публикации. В ячейках — число постов')
plt.xlabel('Период'); plt.ylabel('')
plt.savefig('negative_heatmap.png', dpi=180, bbox_inches='tight')
plt.show()


## 5. Дополнительные гипотезы


In [ ]:
# H2. Негативные посты получают меньше реакций?
negative = df.loc[df['negative'] == 1, 'engagement']
nonnegative = df.loc[df['negative'] == 0, 'engagement']
_, engagement_p = mannwhitneyu(negative, nonnegative, alternative='two-sided')
print('H2:', negative.median(), 'против', nonnegative.median(), 'реакций; вывод — негативные посты обычно менее вовлекающие.')

# H3. Отличается ли длина текста по тональности?
groups = [group['word_count'] for _, group in df.groupby('sent_label', observed=False)]
_, length_p = kruskal(*groups)
print('H3: медианная длина по группам:')
display(df.groupby('sent_label', observed=False)['word_count'].median().reindex(sentiment_order))
print('Вывод — длина текстов различается между тональностями.')

# H4. Зависит ли тональность от формата поста?
content_table = pd.crosstab(df['Content Type'], df['sent_label']).reindex(columns=sentiment_order, fill_value=0)
_, content_p, _, _ = chi2_contingency(content_table)
print('H4: форматы постов имеют разное распределение тональности; связь есть, но формат не объясняет всё.')

# H5. Отличаются ли авторы по доле негатива?
author_rates = df.groupby('Author ID', observed=False).agg(posts=('negative', 'size'), negative_share=('negative', 'mean'))
author_rates = author_rates[author_rates['posts'] >= 30].sort_values('negative_share')
print(f'H5: среди авторов с 30+ постами доля негатива меняется от {author_rates.iloc[0].negative_share:.1%} до {author_rates.iloc[-1].negative_share:.1%}.')

# H6. Является ли понедельник вечером лидером среди 28 сочетаний дня и периода?
cells = df.groupby(['day', 'period'], observed=False).agg(posts=('negative', 'size'), negative_share=('negative', 'mean')).reset_index()
cells['rank'] = cells['negative_share'].rank(method='min', ascending=False).astype(int)
monday_rank = int(cells.query("day == 'Понедельник' and period == 'Вечер'")['rank'].iloc[0])
print(f'H6: понедельник вечером занимает {monday_rank}-е место из {len(cells)}; лидером он не является.')


## 6. Сохраняем результаты

Сохраняются исходный файл с метками, сводные таблицы и основные графики. Эти файлы можно скачать из боковой панели Colab и положить в `results/`.


In [ ]:
from google.colab import files
from pathlib import Path

out = Path('results')
out.mkdir(exist_ok=True)
sentiment_summary.to_csv(out / 'sentiment_summary.csv')
weekday_summary.to_csv(out / 'weekday_summary.csv')
cells.to_csv(out / 'day_period_summary.csv', index=False)
content_table.to_csv(out / 'content_type_summary.csv')
author_rates.to_csv(out / 'author_summary.csv')
pd.DataFrame({
    'hypothesis': ['Понедельник вечером самый негативный', 'Негативные посты получают меньше реакций', 'Длина текста связана с тональностью', 'Формат связан с тональностью', 'Авторы отличаются по доле негатива', 'Понедельник вечером лидер среди 28 ячеек'],
    'conclusion': ['Не подтверждается', 'Поддерживается', 'Поддерживается', 'Связь есть', 'Наблюдается', 'Не подтверждается']
}).to_csv(out / 'hypotheses.csv', index=False)
df.to_csv(out / 'vk_with_sentiment.csv', index=False)
print('Сохранено в папку results')


## Итог

В этой выборке исходная гипотеза не подтверждается. Дополнительный анализ показывает, что негативные посты обычно получают меньше реакций, длина текста и формат связаны с тональностью, а авторы сильно различаются по доле негатива.

Ограничения: тональность определяет модель, часовой пояс в данных не указан, а связи не доказывают причинность.
